In [2]:



import gurobipy as gp
from gurobipy import Model, GRB, quicksum
import numpy as np
import time

options = {
            "WLSACCESSID":"#YOURID",
            "WLSSECRET":"#YOURID",
            "LICENSEID": #YOURID
}


def bounded_beam_search(w, h, n):
    """
    Fungsi heuristik untuk menghasilkan solusi awal feasible.
    """
    solution = np.zeros((w, h), dtype=int)
    blocks = np.random.permutation(n) + 1
    
    # Distribusi blok yang lebih merata
    blocks_per_stack = n // w
    remaining_blocks = n % w
    
    idx = 0
    for j in range(w):
        blocks_this_stack = blocks_per_stack + (1 if j < remaining_blocks else 0)
        for i in range(min(blocks_this_stack, h)):  # Pastikan tidak melebihi tinggi
            if idx >= n:
                break
            solution[j, i] = blocks[idx]
            idx += 1
            
    print("Solusi awal (Bounded Beam Search - Randomized):")
    print(solution.T)
    return solution

def prepare_instance_set(set_type):
    """
    Fungsi untuk mempersiapkan parameter instance dari Set1 atau Set2.
    """
    if set_type == 'Set1':
        w_values = [3, 4, 5, 6, 10]
        h_values = range(5, 13)  # h = 5 hingga 12
        n = lambda w, h: w * (h - 2)
    elif set_type == 'Set2':
        w_values = [6, 7, 8, 9, 10]
        h_values = range(3, 7)  # h = 3 hingga 6
        # Memastikan n valid untuk Set2
        def n(w, h):
            min_n = (w - 1) * h
            max_n = w * h - 1
            n_val = np.random.randint(min_n, max_n + 1)
            return min(n_val, w * h - 1)  
    else:
        raise ValueError("Invalid set_type. Must be 'Set1' or 'Set2'")
    
    return w_values, h_values, n

def get_stack(initial_solution, block):
    """Helper function to get the initial stack of a block"""
    pos = np.where(initial_solution == block)
    return pos[0][0] if len(pos[0]) > 0 else -1

def get_pi_value(block_id, initial_solution):
    stack_index = -1
    height_index = -1

    # Cari posisi blok
    for j in range(initial_solution.shape[0]):  # stack
        for h in range(initial_solution.shape[1]):  # height
            if initial_solution[j, h] == block_id + 1:
                stack_index, height_index = j, h
                break

    if stack_index == -1:
        return initial_solution.shape[1]  # Blok tidak ditemukan

    # Hitung π_i
    pi_value = min(initial_solution[stack_index, :height_index + 1])
    return pi_value - 1  # Sesuaikan indeks (0-based)

def calculate_phi(n, pi, s, q, w):
    """
    Calculate φi,t and φi,j,t according to theoretical definitions.
    """
    phi_it = {i: {} for i in range(n)}
    phi_ijt = {i: {} for i in range(n)}

    for i in range(n):
        # Only calculate up to i, not n+1
        for t in range(pi[i] + 1, i + 1):
            # Calculate φi,t
            if t == pi[i] + 1:
                # Special case for t = πi + 1
                phi_it[i][t] = [
                    i_prime for i_prime in range(n)
                    if (pi[i] + 1 <= i_prime < i and pi[i_prime] < pi[i]) or  # First set
                       (pi[i] + 1 <= i_prime < i and pi[i_prime] == pi[i] and q[i_prime] > q[i])  # Second set
                ]
            else:
                # General case for t > πi + 1
                phi_it[i][t] = [
                    i_prime for i_prime in range(n)
                    if t <= i_prime < i and pi[i_prime] < t
                ]

            # Calculate φi,j,t for each stack
            phi_ijt[i][t] = {}
            for j in range(w):
                phi_ijt[i][t][j] = [
                    i_prime for i_prime in range(n)
                    if t <= i_prime < i and pi[i_prime] >= t and s[i_prime] == j
                ]

    return phi_it, phi_ijt




def branch_and_cut_rbrp(w, h, n, initial_solution):
    """
    Implementasi Branch-and-Cut untuk Restricted Block Relocation Problem.
    """
    env = gp.Env(params=options)
    # Formulate problem


    model = Model("RBRP", env=env )
    model.setParam('TimeLimit', 3600)  # Batas waktu 1 jam
    model.setParam('Threads', 1)       # Single-thread


    # Variables
    x = model.addVars(n, w, n, vtype=GRB.BINARY, name="x") # constraint 11
    y = model.addVars(n, w, n, vtype=GRB.BINARY, name="y") # constraint 12

    # constraints (2)-(7)
    ## constraint 2
    for i in range(n):
        for t in range(i+1):
        #for t in range(n):
            model.addConstr(quicksum(x[i, j, t] for j in range(w)) == 1)
    ## constraint 3
    for j in range(w):
        for t in range(n):
            #model.addConstr(quicksum(x[i, j, t] for i in range(n)) <=h)
            model.addConstr(quicksum(x[i, j, t] for i in range(t, n)) <= h)
    
    ## constraint 4-7
    for i in range(n):
        for j in range(w):
            for t in range(i):
                model.addConstr(y[i, j, t] >= x[i, j, t] - x[i, j, t+1])
                model.addConstr(y[i, j, t] <= 1 - x[i, j, t+1])
                model.addConstr(y[i, j, t] <= x[t, j, t])
                model.addConstr(y[i, j, t] <= x[i, j, t])

    # Initial configuration constraints (10)
    for i in range(n):
        s_i = get_stack(initial_solution, i + 1)
        if s_i >= 0:            
                model.addConstr(x[i, s_i, 0] == 1)

    # Hitung nilai π_i dan stack awal s_i
    pi = {i: get_pi_value(i, initial_solution) for i in range(n)}
    s = {i: get_stack(initial_solution, i + 1) for i in range(n)}
    q = {i: i for i in range(n)}  # Assign priority/order as block index
    phi_it, phi_ijt = calculate_phi(n, pi, s, q, w)


    # Constraint (13): Blok tidak boleh dipindahkan jika π_i = i

    for i in range(n):
        for t in range(1, pi[i]):
            model.addConstr(x[i, s[i], t] == 1)  # Enforcing that the block remains in its initial stack until its reshuffling time.

    # Constraint 15
    for i in range(n):
        for j in range(w):
            if pi[i] + 1 in phi_it[i]:  # Check if the starting time exists
                min_value = min(
                    {i} | 
                    set(phi_it[i][pi[i] + 1]) | 
                    set(phi_ijt[i][pi[i] + 1][j])
                )
                # Directly use the correct range
                for t in range(pi[i] + 1, min_value):
                    model.addConstr(x[i, j, t] <= x[i, j, t + 1])

    # Constraint 16
    for i in range(n):
        if pi[i] + 1 in phi_it[i]:  # Check if the starting time exists
            min_value = min(
                {i} | 
                set(phi_it[i][pi[i] + 1]) | 
                set().union(*[set(phi_ijt[i][pi[i] + 1][j]) for j in range(w)])
            )
            # Directly use the correct range
            for t in range(pi[i] + 1, min_value):
                model.addConstr(
                    quicksum(x[i, j, t] for j in range(w)) == 
                    quicksum(x[i, j, t + 1] for j in range(w))
                )

    # Constraint 17
    for i in range(n):
        t = pi[i] + 1
        if t in phi_it[i]:
            for j in range(w):
                if j != s[i]:  # Check j≠s(i)
                    if j in phi_ijt[i][t] and phi_ijt[i][t][j]:  # Check φi,j,πi+1 ≠ ∅
                        min_phi_ij = min(phi_ijt[i][t][j])
                        
                        model.addConstr(
                            x[i, j, pi[i] + 1] <= 
                            y[i, j, min_phi_ij] +  # Single term yi,j,min(φi,j,πi+1)
                            quicksum(  # Sum over i'∈φi,πi+1 where i'<min(φi,j,πi+1)
                                y[i_prime, j, i_prime]
                                for i_prime in phi_it[i][t]
                                if i_prime < min_phi_ij
                            )
                        )

    # Constrain 18                                    
    for i in range(n):
        for t in range(pi[i] + 1, i + 1):  # t = πi + 1,...,i
            for j in range(w):
                # Get all i' ∈ φi,t (blocks reshuffled before time t)
                for i_prime in phi_it[i][t]:
                    # Check if min(φi,j,t) > i'
                    if j in phi_ijt[i][t] and phi_ijt[i][t][j]:
                        min_phi_ijt = min(phi_ijt[i][t][j])
                        if min_phi_ijt > i_prime:
                            model.addConstr(
                                y[i, j, i_prime] + 
                                quicksum(y[i, j, i_double_prime] 
                                    for i_double_prime in phi_it[i][t])  # sum over φi,t
                                >= x[i, j, t] + x[i_prime, j, t - 1] - x[i, j, t - 1] - 1
                            )
    # Objective function
    model.setObjective(quicksum(y[i, j, t]
                               for i in range(n)
                               for j in range(w)
                               for t in range(i)),
                      GRB.MINIMIZE)
    
    # Start the timer
    start_time = time.time()
    model.setParam('OutputFlag', 1)
    model.setParam('LogToConsole', 1)

    model.optimize()
    end_time = time.time()

    optimization_time = end_time - start_time


    total_instances = sum(1 for _ in range(n))
    solved_instances = sum(1 for i in range(n) if model.status == GRB.OPTIMAL)
        
    # Output solution
    if model.status == GRB.OPTIMAL or model.status == GRB.TIME_LIMIT:
        opt_status = solved_instances
        avg_time_solved = optimization_time
        avg_cuts = model.getAttr(GRB.Attr.NumConstrs)    
        avg_gap_final = 0  # Fully solved

        ## hapus pada bagian "#" sebelum """ di bawah ini apabila tidak ingin menampilkan langkah-langkah perpindahannya
        #"""
        print("Solution found:")
        solution_matrix = np.zeros((n, w, h), dtype=int)

        for i in range(n):
            for j in range(w):
                for t in range(n):
                    if x[i, j, t].x > 0.5:
                        # Temukan ketinggian pertama yang kosong di stack j pada waktu t
                        height = np.argmax(solution_matrix[t, j, :] == 0)
                        solution_matrix[t, j, height] = i + 1
    
        # Tampilkan solusi per waktu
        for t in range(n):
            print(f"Time {t + 1}:")
            print(solution_matrix[t].T)  # Transpose untuk visualisasi vertikal
        #"""            
    else:
        print("No feasible solution found.")
        opt_status = f"{solved_instances}/{total_instances}" 
        avg_time_solved = '-'
        avg_cuts = '-'
        avg_gap_final = model.MIPGap * 100 if model.MIPGap is not None else '-'        

    # Calculate statistics
    stats = {
        'w': w,
        'h': h,
        'n': n,
        'I' : sum(1 for i in range(n)),
        'Opt': opt_status,        
        'Vars': model.numVars,
        'Cons': model.numConstrs,
        'Glp': model.ObjVal if model.status == GRB.OPTIMAL else 0.0,
        'Tlp': optimization_time,
        'Nodes': model.NodeCount, #model.BarIterCount
        'TC': optimization_time,  # Total computation time
        'T': avg_time_solved,
        'Cuts': avg_cuts,
        'Gf': avg_gap_final        
    }
    
    return stats

# Menyiapkan instance dan menjalankan optimasi
all_stats = []
w_values, h_values, n_function = prepare_instance_set("Set1") # ganti set 2 apabila diperlukan
for w in w_values:
    for h in h_values:
        
        n = n_function(w, h)
        initial_solution = bounded_beam_search(w, h, n)
        stats = branch_and_cut_rbrp(w, h, n, initial_solution) 
        all_stats.append(stats)
        print(stats)
    
# Print out the stats
print("\nOverall Statistics Summary:")
for stat in all_stats:
    print(stat)



Solusi awal (Bounded Beam Search - Randomized):
[[4 6 9]
 [2 1 8]
 [3 7 5]
 [0 0 0]
 [0 0 0]]
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2602095
Academic license 2602095 - for non-commercial use only - registered to 60___@student.its.ac.id
Set parameter TimeLimit to value 3600
Set parameter Threads to value 1
Set parameter OutputFlag to value 1
Set parameter LogToConsole to value 1
Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: AMD Ryzen 9 6900HS with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  3600
Threads  1

Academic license 2602095 - for non-commercial use only - registered to 60___@student.its.ac.id
Optimize a model with 547 rows, 486 columns and 1303 nonzeros
Model fingerprint: 0x700367d6
Variable types: 0 continuous, 486 integer (486 binary)
Coefficient statistics:
  Matrix ran